# Notebook 00 — Context

**Continual Learning in Context: RML Extension for CL-BENCH**

This notebook initializes the RML extension.

It is designed to work in both:

- **GitHub / local repo execution**
- **Google Colab**, including when opened directly from the Colab badge

Notebook 00 establishes the baseline workflow:

1. Locate or clone the repo.
2. Import the RML `src/` helpers.
3. Create a toy stateful/stateless reward trajectory.
4. Compute gain, stability, plasticity, and drift flags.
5. Save figures, CSV, JSON, and a downloadable artifact archive.

## 0. Bootstrap Repo Path

Run this first.

If the notebook is running in Colab and the repo is not available, this cell clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

Then it moves into the `rml/` folder and adds it to `sys.path`.

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    candidates = [start, *start.parents]

    for base in candidates:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"

    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("src/gain.py exists:", (RML_ROOT / "src" / "gain.py").exists())
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain, cumulative_gain, average_gain
from src.stability import compute_stability, compute_plasticity
from src.drift import detect_drift
from src.utils import ensure_directory

print("Imports complete.")

## 2. Context

CL-BENCH evaluates whether AI systems improve through sequential experience.

The benchmark separates:

- **reward**: how well a system performs on each task instance
- **gain**: how much the stateful system improves over the same system run statelessly

The core comparison is:

\[
g_t = r_t^{sf} - r_t^{sl}
\]

where:

- \(r_t^{sf}\) is reward from the stateful system at instance \(t\)
- \(r_t^{sl}\) is reward from the stateless baseline at instance \(t\)

RML adds an interpretability layer:

> What context was reused, when did it help, and when did it become stale?

## 3. Toy Trajectory

Notebook 00 uses a small synthetic trajectory so the extension runs immediately.

Later notebooks can replace this with real CL-BENCH rollout logs.

In [ ]:
instances = np.arange(1, 13)

stateful_rewards = np.array([
    0.18, 0.22, 0.28, 0.35,
    0.43, 0.48, 0.46, 0.52,
    0.40, 0.45, 0.55, 0.60
])

stateless_rewards = np.array([
    0.18, 0.19, 0.20, 0.21,
    0.22, 0.20, 0.21, 0.22,
    0.23, 0.22, 0.24, 0.25
])

variants = [
    "A", "A", "A", "A",
    "B", "B", "B", "B",
    "C", "C", "C", "C"
]

df = pd.DataFrame({
    "instance": instances,
    "variant": variants,
    "stateful_reward": stateful_rewards,
    "stateless_reward": stateless_rewards,
})

df

## 4. Gain

Gain measures the value of accumulated state relative to a fresh stateless run on the same instance.

In [ ]:
df["gain"] = compute_gain(
    df["stateful_reward"].tolist(),
    df["stateless_reward"].tolist(),
)

summary = {
    "cumulative_gain": float(cumulative_gain(df["gain"].tolist())),
    "average_gain": float(average_gain(df["gain"].tolist())),
    "mean_stateful_reward": float(df["stateful_reward"].mean()),
    "mean_stateless_reward": float(df["stateless_reward"].mean()),
}

summary

## 5. Stability and Plasticity

For this toy schedule, variant boundaries are:

```python
[0, 4, 8]
```

Interpretation:

- **stability**: gain at variant boundaries, where older structure must transfer
- **plasticity**: gain away from boundaries, where the system adapts within a variant

In [ ]:
variant_boundaries = [0, 4, 8]

summary["stability"] = float(
    compute_stability(df["gain"].tolist(), variant_boundaries)
)

summary["plasticity"] = float(
    compute_plasticity(df["gain"].tolist(), variant_boundaries)
)

summary

## 6. Drift / Stale Context Flags

A simple first-pass detector flags negative gain.

In real CL-BENCH logs, negative gain can indicate:

- stale memory reuse
- overfitting to recent observations
- failure to detect concept drift
- harmful context compression

In [ ]:
drift_indices = detect_drift(df["gain"].tolist(), threshold=-0.02)

df["possible_drift_or_stale_context"] = False
df.loc[drift_indices, "possible_drift_or_stale_context"] = True

df

## 7. Figure — Stateful vs. Stateless Reward

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df["instance"],
    df["stateful_reward"],
    marker="o",
    label="Stateful",
)

ax.plot(
    df["instance"],
    df["stateless_reward"],
    marker="o",
    linestyle="--",
    label="Stateless",
)

for boundary in [5, 9]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 00: Stateful vs. Stateless Reward")
ax.set_xlabel("Instance")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(True, alpha=0.3)

reward_fig_path = FIGURES_DIR / "00_context_stateful_vs_stateless.png"
fig.tight_layout()
fig.savefig(reward_fig_path, dpi=160)

reward_fig_path

## 8. Figure — Gain Trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(df["instance"], df["gain"])
ax.axhline(0, linewidth=1)

for boundary in [5, 9]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 00: Gain Trajectory")
ax.set_xlabel("Instance")
ax.set_ylabel("Gain = Stateful - Stateless")
ax.grid(True, axis="y", alpha=0.3)

gain_fig_path = FIGURES_DIR / "00_context_gain.png"
fig.tight_layout()
fig.savefig(gain_fig_path, dpi=160)

gain_fig_path

## 9. Export Artifacts

Notebook 00 saves:

- `results/00_context_gain.csv`
- `results/00_context_summary.json`
- `figures/00_context_stateful_vs_stateless.png`
- `figures/00_context_gain.png`
- `results/00_context_artifacts.zip`

In [ ]:
csv_path = RESULTS_DIR / "00_context_gain.csv"
json_path = RESULTS_DIR / "00_context_summary.json"
zip_path = RESULTS_DIR / "00_context_artifacts.zip"

df.to_csv(csv_path, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "00_context.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(json_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [csv_path, json_path, reward_fig_path, gain_fig_path]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [csv_path, json_path, reward_fig_path, gain_fig_path, zip_path]:
    print("-", path)

## 10. Download Artifacts

In Colab, the next cell downloads the zip file.

Locally, the archive is saved in:

```text
rml/results/00_context_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 11. Notebook 00 Claim

CL-BENCH asks whether systems improve through sequential experience.

RML adds:

> Continual learning is not merely memory. It is context reuse under constraint.

Notebook 00 establishes the minimal workflow:

\[
\text{reward}
\rightarrow
\text{gain}
\rightarrow
\text{stability/plasticity}
\rightarrow
\text{drift flags}
\]

Later notebooks refine this with real CL-BENCH rollout logs and the eight-lane lab-report structure:

\[
0 \rightarrow \{1,7,11,13,17,19,23,29\}
\]